# Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque
import random
import copy
import csv

N = 12   # number of users
C = 30   # max queue size per user

# Arrival probabilities for each user
arrival_probs = [0.55, 0.65, 0.80, 0.55, 0.65, 0.80, 0.50, 0.65, 0.80, 0.45, 0.60, 0.75] 

# Discount factor and drop multiplier
gamma = 0.95
lamb = 2

# Define the channel states, service rates and number of actions
service_rate_levels = [1, 2, 3, 4, 6, 8, 10, 12]
K = len(service_rate_levels)
max_service_rate = max(service_rate_levels)
actions = list(range(N))

# Evaluation parameters
evaluation_horizon = 500
num_train_episodes = 500
evaluation_every = num_train_episodes
num_eval_episodes = 1000
training_horizon_DQN = 200

# Load channel trace from CSV file
def load_rates_csv(filename):

    rows = []

    with open(filename, "r") as f:
        reader = csv.reader(f)
        next(reader) 

        for row in reader:
            rates_for_slot = [int(x) for x in row[1:]]  # ignore slot column
            rows.append(rates_for_slot)

    rates = np.array(rows, dtype=np.int32)
    return rates

# Sample starting slot from trace
def sample_start_slot(rates, horizon):

    T = rates.shape[0]   # number of rows / slots in CSV

    max_start = T - horizon

    start_slot = np.random.randint(0, max_start + 1)

    return start_slot

# Get service rate from trace
def get_service_rate_from_trace(rates, start_slot, user_id, t):
 
    trace_slot = start_slot + t

    return int(rates[trace_slot, user_id])

# Load the channel trace
rates = load_rates_csv("synthetic_12_user_service_rates.csv")

# Set seed
seed = 46
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# DQN

In [ ]:
replay_buffer = deque(maxlen=10000)
batch_size = 32

target_update_every = 100
training_step = 0

# Define the neural network architecture for DQN
input_dim = 3*N
output_dim = len(actions)

Q_online = nn.Sequential(
    nn.Linear(input_dim, 32),
    nn.ReLU(),
    nn.Linear(32, output_dim)
)

Q_target = copy.deepcopy(Q_online)
Q_target.eval()

optimizer = optim.Adam(Q_online.parameters(), lr=0.001)

# State to tensor conversion function
def state_to_tensor(state):
    values = []

    for user_id in range(N):
        queue, channel = state[user_id]

        # normalized queue length
        values.append(float(queue) / C)

        # normalized channel state
        values.append(float(channel) / max_service_rate)

        # normalized throughput
        values.append(min(float(queue), float(channel)) / max_service_rate)

    return torch.tensor(values, dtype=torch.float32).unsqueeze(0)

# Best action selection
def best_action_DQN(state):
    with torch.no_grad():
        x = state_to_tensor(state)
        q_values = Q_online(x)

    best_action_index = torch.argmax(q_values).item()
    return actions[best_action_index]

# Run an episode for DQN
def run_episode_DQN(horizon, epsilon, training_mode):

    global training_step
    total_episode_reward = 0
    total_episode_throughput = 0
    total_episode_drops = 0
    total_episode_queue_occupancy_per_user = np.zeros(N)

    rewards = np.zeros(horizon)
    throughputs = np.zeros(horizon)
    drops = np.zeros(horizon)

    # Initial state for each user: (queue_length, channel_state)
    # Pick random starting slot for each user
    start_slot = sample_start_slot(rates, horizon + 1)

    state = {}

    # Choose initial queue lengths and service rates for each user
    for user_id in range(N):
        initial_queue = np.random.randint(0, C // 2 + 1)  # 0 to 15 when C = 30

        initial_service_rate = rates[start_slot, user_id]
        state[user_id] = (initial_queue, initial_service_rate)

    for t in range(horizon):

        queues = [state[user_id][0] for user_id in range(N)]
        channels = [state[user_id][1] for user_id in range(N)]

        # Action selection
        if training_mode and np.random.random() < epsilon:
            action = np.random.choice(actions)
        else:
            action = best_action_DQN(state)

        # Transmission
        service_rate = channels[action]
        packets_to_send = min(queues[action], service_rate)

        queues[action] -= packets_to_send

        # Arrivals and drops
        dropped_packets = 0

        for i in range(N):
            if np.random.random() < arrival_probs[i]:
                if queues[i] < C:
                    queues[i] += 1
                else:
                    dropped_packets += 1
    
        # Reward
        reward = packets_to_send - lamb * dropped_packets

        # Track reward, throughput and drops
        total_episode_reward += reward
        total_episode_throughput += packets_to_send
        total_episode_drops += dropped_packets

        rewards[t] = reward
        throughputs[t] = packets_to_send
        drops[t] = dropped_packets

        # Track queue occupancy per user
        total_episode_queue_occupancy_per_user += np.array(queues)

        # Update channel states
        next_channels = []

        for i in range(N):
            next_channel = rates[start_slot + t + 1, i]
            next_channels.append(next_channel)

        # Build next state
        next_state = {
            user_id: (queues[user_id], next_channels[user_id])
            for user_id in range(N)
        }

        # Store transition in replay buffer (only in training mode)
        state_tensor = state_to_tensor(state).squeeze(0)
        next_state_tensor = state_to_tensor(next_state).squeeze(0)

        if training_mode:
            replay_buffer.append((state_tensor, action, reward, next_state_tensor))

        # Sample a random mini-batch from the replay buffer
        if training_mode and len(replay_buffer) >= batch_size:
            batch = random.sample(replay_buffer, batch_size)

            # Prepare tensors for the batch
            state_batch = torch.stack([transition[0] for transition in batch])
            action_batch = torch.tensor([transition[1] for transition in batch], dtype=torch.long).unsqueeze(1)
            reward_batch = torch.tensor([transition[2] for transition in batch], dtype=torch.float32)
            next_state_batch = torch.stack([transition[3] for transition in batch])

            # Online network predicts Q_online(s, a)
            q_values = Q_online(state_batch)
            current_q_values = q_values.gather(1, action_batch).squeeze(1)

            # Build targets using target network
            with torch.no_grad():
                next_q_values = Q_target(next_state_batch)
                max_next_q_values = torch.max(next_q_values, dim=1)[0]

                target_q_values = reward_batch + gamma * max_next_q_values

            # Compute loss and update online network
            loss = F.mse_loss(current_q_values, target_q_values)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Periodically update target network
            training_step += 1

            if training_step % target_update_every == 0:
                Q_target.load_state_dict(Q_online.state_dict())
            
        state = next_state
    
    return (
        total_episode_reward,
        total_episode_throughput,
        total_episode_drops,
        total_episode_queue_occupancy_per_user,
        rewards,
        throughputs,
        drops
    )


def DQN_learning(episodes, epsilon, epsilon_min, epsilon_decay):

    total_training_time = 0.0
    total_evaluation_time = 0.0
    
    episode_rewards = []

    evaluation_points = []
    evaluation_rewards = []
    evaluation_throughputs = []
    evaluation_drops = []
    evaluation_epsilons = []
    evaluation_queue_occupancies_per_user = []
    evaluation_queue_occupancies_all_users = []

    evaluation_rewards_per_slot = []
    evaluation_throughputs_per_slot = []
    evaluation_drops_per_slot = []

    for ep in range(episodes):

        train_start = time.time()

        total_episode_reward, total_episode_throughput, total_episode_drops, total_episode_queue_occupancy_per_user, _, _, _ = run_episode_DQN(
            horizon=training_horizon_DQN,
            epsilon=epsilon,
            training_mode=True
        )

        train_end = time.time()
        total_training_time += train_end - train_start

        episode_rewards.append(total_episode_reward)

        # Decay exploration
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

        # Evaluation every evaluation_every training episodes
        if (ep + 1) % evaluation_every == 0:

            eval_start = time.time()

            eval_rewards = []
            eval_throughputs = []
            eval_drops = []
            eval_queue_occupancies_per_user = []

            eval_rewards_per_slot = np.zeros((num_eval_episodes, evaluation_horizon))
            eval_throughputs_per_slot = np.zeros((num_eval_episodes, evaluation_horizon))
            eval_drops_per_slot = np.zeros((num_eval_episodes, evaluation_horizon))

            for eval_ep in range(num_eval_episodes):

                total_reward, total_throughput, total_drops, total_queue_occupancy_per_user, rewards, throughputs, drops = run_episode_DQN(
                    horizon=evaluation_horizon,
                    epsilon=epsilon,
                    training_mode=False
                )

                eval_rewards.append(total_reward / evaluation_horizon)
                eval_throughputs.append(total_throughput / evaluation_horizon)
                eval_drops.append(total_drops / evaluation_horizon)
                eval_queue_occupancies_per_user.append(total_queue_occupancy_per_user / evaluation_horizon)

                eval_rewards_per_slot[eval_ep] = rewards
                eval_throughputs_per_slot[eval_ep] = throughputs
                eval_drops_per_slot[eval_ep] = drops

            eval_end = time.time()
            total_evaluation_time += eval_end - eval_start
            
            evaluation_rewards_per_slot.append(eval_rewards_per_slot.mean(axis=0))
            evaluation_throughputs_per_slot.append(eval_throughputs_per_slot.mean(axis=0))
            evaluation_drops_per_slot.append(eval_drops_per_slot.mean(axis=0))

            evaluation_points.append(ep + 1)
            evaluation_rewards.append(np.mean(eval_rewards))
            evaluation_throughputs.append(np.mean(eval_throughputs))
            evaluation_drops.append(np.mean(eval_drops))

            mean_queue_occupancy_per_user = np.mean(eval_queue_occupancies_per_user, axis=0)
            mean_queue_occupancy_all_users = np.mean(mean_queue_occupancy_per_user)

            evaluation_queue_occupancies_per_user.append(mean_queue_occupancy_per_user)
            evaluation_queue_occupancies_all_users.append(mean_queue_occupancy_all_users)

            evaluation_epsilons.append(epsilon)

            print(
                "DQN evaluation at training episode",
                ep + 1,
                "| epsilon =",
                epsilon,
                "| average reward per slot =",
                evaluation_rewards[-1],
                "| average throughput per slot =",
                evaluation_throughputs[-1],
                "| average drops per slot =",
                evaluation_drops[-1],
                "| mean queue occupancy all users =",
                evaluation_queue_occupancies_all_users[-1],
                "| mean queue occupancy per user =",
                evaluation_queue_occupancies_per_user[-1],
                flush=True
            )

        print("DQN training episode", ep + 1, flush=True)

    return (
        episode_rewards,
        evaluation_points,
        evaluation_rewards,
        evaluation_throughputs,
        evaluation_drops,
        evaluation_queue_occupancies_per_user,
        evaluation_queue_occupancies_all_users,
        evaluation_epsilons,
        evaluation_rewards_per_slot,
        evaluation_throughputs_per_slot,
        evaluation_drops_per_slot,
        total_training_time,
        total_evaluation_time
    )

# Run DQN
start_dqn = time.time()

(
    episode_rewards_DQN,
    evaluation_points_DQN,
    evaluation_rewards_DQN,
    evaluation_throughputs_DQN,
    evaluation_drops_DQN,
    evaluation_queue_occupancies_per_user_DQN,
    evaluation_queue_occupancies_all_users_DQN,
    evaluation_epsilons_DQN,
    evaluation_rewards_per_slot_DQN,
    evaluation_throughputs_per_slot_DQN,
    evaluation_drops_per_slot_DQN,
    dqn_training_time,
    dqn_evaluation_time
) = DQN_learning(
    episodes=num_train_episodes,
    epsilon=1,
    epsilon_min=0.05,
    epsilon_decay= 0.99204320482
)

end_dqn = time.time()
dqn_total_time = end_dqn - start_dqn

print("\nDQN training time: {:.2f} seconds".format(dqn_training_time))
print("DQN evaluation time: {:.2f} seconds".format(dqn_evaluation_time))
print("DQN total wall-clock time: {:.2f} seconds".format(dqn_total_time))

# Baselines

In [ ]:
# Longest Queue: schedule the user with the largest queue
def longest_queue_policy(state):
    """
    Selects a user based on:
    1. Longest queue
    2. Random tie-breaker
    """

    queues = [state[user_id][0] for user_id in range(N)]

    max_q_value = max(queues)

    max_q_users = [
        i for i, q in enumerate(queues)
        if q == max_q_value
    ]

    return np.random.choice(max_q_users)

# Max-Weight: schedule the user with largest queue-channel product
def max_weight_policy(state):
    """
    Selects a user based on:
    1. Largest q_i * r_i
       where q_i is queue length and r_i is the service rate of the current channel
    2. Random tie-breaker
    """

    scores = []

    for user_id in range(N):
        queue, channel = state[user_id]

        if queue == 0:
            scores.append(-np.inf)   # avoid serving empty queues
        else:
            rate = channel
            scores.append(queue * rate)

    # If all queues are empty, choose randomly
    if max(scores) == -np.inf:
        return np.random.choice(N)

    max_score = max(scores)

    max_weight_users = [
        i for i, s in enumerate(scores)
        if s == max_score
    ]

    return np.random.choice(max_weight_users)

# Greedy Throughput: schedule the user that can transmit the most packets immediately
def greedy_throughput_policy(state):
    """
    Selects a user based on:
    1. Largest immediate throughput: min(queue_i, service_rate_i)
    2. Random tie-breaker
    """

    scores = []

    for user_id in range(N):
        queue, channel = state[user_id]

        service_rate = channel

        # actual packets that can be transmitted now
        score = min(queue, service_rate)

        scores.append(score)

    max_score = max(scores)

    # If all queues are empty, choose randomly
    if max_score == 0:
        return np.random.choice(N)

    best_users = [
        i for i, s in enumerate(scores)
        if s == max_score
    ]

    return np.random.choice(best_users)

# Best Non-Empty Channel: schedule the non-empty user with the best current channel
def best_nonempty_channel_policy(state):
    """
    Selects a user based on:
    1. Queue must be non-empty
    2. Best channel/service rate among non-empty users
    3. Random tie-breaker
    """

    nonempty_users = [
        i for i in range(N)
        if state[i][0] > 0
    ]

    # If all queues are empty, choose randomly
    if len(nonempty_users) == 0:
        return np.random.choice(N)

    scores = []

    for i in nonempty_users:
        queue, channel = state[i]
        scores.append(channel)

    max_score = max(scores)

    best_users = [
        user for user, score in zip(nonempty_users, scores)
        if score == max_score
    ]

    return np.random.choice(best_users)

# Simulation

In [ ]:
def simulate_episode(policy_func, horizon):

    # Initial state for each user: (queue_length, channel_state)
    # Pick random starting slot for each user
    start_slot = sample_start_slot(rates, horizon + 1)

    state = {}

    # Choose initial queue lengths and service rates for each user
    for user_id in range(N):
        initial_queue = np.random.randint(0, C // 2 + 1)  # 0 to 15 when C = 30

        initial_service_rate = rates[start_slot, user_id]
        state[user_id] = (initial_queue, initial_service_rate)

    discounted_return = 0.0

    current_gamma = 1.0

    rewards = np.zeros(horizon)
    throughputs = np.zeros(horizon)
    drops = np.zeros(horizon)

    for t in range(horizon):

        # unpack current state
        queues = [state[user_id][0] for user_id in range(N)]
        channels = [state[user_id][1] for user_id in range(N)]

        # 1. Pick user
        selected_user = policy_func(state)

        # 2. Transmission
        service_rate = channels[selected_user]
        packets_to_send = min(queues[selected_user], service_rate)

        queues[selected_user] -= packets_to_send

        # 3. Arrivals and drops
        dropped_packets = 0

        for i in range(N):
            if np.random.random() < arrival_probs[i]:
                if queues[i] < C:
                    queues[i] += 1
                else:
                    dropped_packets += 1

        # 4. Reward calculation and discounting
        reward = packets_to_send - lamb * dropped_packets

        discounted_return += current_gamma * reward

        rewards[t] = reward
        throughputs[t] = packets_to_send
        drops[t] = dropped_packets

        # 5. Update channel states
        next_channels = []

        for i in range(N):
            next_channel = rates[start_slot + t + 1, i]
            next_channels.append(next_channel)

        # 6. Build next state
        state = {
            user_id: (queues[user_id], next_channels[user_id])
            for user_id in range(N)
        }

        # 7. Discount update
        current_gamma *= gamma

    return discounted_return, rewards, throughputs, drops

# Run policy simulations
num_episodes = 3000

def run_policy(policy_func, policy_name, horizon):
    returns = []
    rewards_all = np.zeros((num_episodes, horizon))
    throughputs_all = np.zeros((num_episodes, horizon))
    drops_all = np.zeros((num_episodes, horizon))

    print(f"\n{policy_name}")
    print("-" * 20)
    print()

    for i in range(num_episodes):
        print(f"\r{policy_name}: Episode {i+1}/{num_episodes}", end="", flush=True)

        ret, rewards, throughputs, drops = simulate_episode(
            policy_func,
            horizon=horizon
        )

        returns.append(ret)
        rewards_all[i] = rewards
        throughputs_all[i] = throughputs
        drops_all[i] = drops

    print()

    mean_return = np.mean(returns)
    mean_rewards = rewards_all.mean(axis=0)
    mean_throughputs = throughputs_all.mean(axis=0)
    mean_drops = drops_all.mean(axis=0)

    print(f"{policy_name} mean discounted return:", mean_return)
    print(f"{policy_name} mean reward per round:", np.mean(mean_rewards))
    print(f"{policy_name} mean throughput per round:", np.mean(mean_throughputs))
    print(f"{policy_name} mean drops per round:", np.mean(mean_drops))
    print()

    return mean_return, mean_rewards, mean_throughputs, mean_drops

mean_return_lq, mean_rewards_lq, mean_throughputs_lq, mean_drops_lq = run_policy(
    longest_queue_policy,
    "Longest-queue",
    evaluation_horizon
)

mean_return_mw, mean_rewards_mw, mean_throughputs_mw, mean_drops_mw = run_policy(
    max_weight_policy,
    "Max-weight",
    evaluation_horizon
)

mean_return_gt, mean_rewards_gt, mean_throughputs_gt, mean_drops_gt = run_policy(
    greedy_throughput_policy,
    "Greedy-throughput",
    evaluation_horizon
)

mean_return_bnec, mean_rewards_bnec, mean_throughputs_bnec, mean_drops_bnec = run_policy(
    best_nonempty_channel_policy,
    "Best-nonempty-channel",
    evaluation_horizon
)

# Lookahead Beam Search 

In [ ]:

lookahead_horizon = 8   # L
beam_widths = [12, 36, 108 , 324, 324, 324, 324, 324]   # B
num_episodes_beam = 200 # start small; increase to 1000 if runtime is okay

def simulate_expected_step(sim_queues, action, channels):
    """
    Simulates one future slot for the beam search.
    Uses expected arrivals, not random arrivals.
    """

    new_queues = sim_queues.copy()

    # Transmission
    service_rate = channels[action]
    packets_to_send = min(new_queues[action], service_rate)

    new_queues[action] -= packets_to_send

    # Expected arrivals and expected drops
    expected_drops = 0.0

    for i in range(N):
        q_after_arrival = new_queues[i] + arrival_probs[i]

        if q_after_arrival > C:
            expected_drops += q_after_arrival - C
            new_queues[i] = C
        else:
            new_queues[i] = q_after_arrival

    # Reward used inside lookahead
    reward = packets_to_send - lamb * expected_drops

    return new_queues, reward


def beam_oracle_action(state, start_slot, t, remaining_slots, lookahead_horizon, beam_widths):
    """
    Beam-search lookahead oracle.

    It uses future service rates from the CSV trace.
    It uses expected arrivals inside the simulated future.
    It returns only the first action of the best sequence.
    """

    # Current queues
    initial_queues = np.array(
        [state[user_id][0] for user_id in range(N)],
        dtype=float
    )

    # Each beam candidate is:
    # (action_sequence, simulated_queues, cumulative_reward)
    beam = [
        ([], initial_queues, 0.0)
    ]

    max_depth = min(lookahead_horizon, remaining_slots)

    for depth in range(max_depth):

        candidates = []

        # Future channel/service rates from trace
        channels = rates[start_slot + t + depth]

        for action_sequence, sim_queues, cumulative_reward in beam:

            for action in actions:

                new_queues, reward = simulate_expected_step(
                    sim_queues,
                    action,
                    channels
                )

                new_action_sequence = action_sequence + [action]

                new_cumulative_reward = cumulative_reward + (gamma ** depth) * reward

                candidates.append(
                    (new_action_sequence, new_queues, new_cumulative_reward)
                )

        # Random shuffle before sorting, just to avoid always choosing the same action in exact ties
        random.shuffle(candidates)

        # Keep only the best beam_width candidates
        candidates.sort(key=lambda x: x[2], reverse=True)

        # Keep only the best candidates for this lookahead depth
        current_beam_width = beam_widths[depth]
        beam = candidates[:current_beam_width]

    # Choose the best sequence and return only its first action
    best_sequence = beam[0][0]

    return best_sequence[0]

def simulate_episode_beam_oracle(horizon, lookahead_horizon, beam_widths):

    # Need extra future slots because the oracle looks ahead
    start_slot = sample_start_slot(rates, horizon + lookahead_horizon)

    state = {}

    for user_id in range(N):
        initial_queue = np.random.randint(0, C // 2 + 1)
        initial_service_rate = rates[start_slot, user_id]
        state[user_id] = (initial_queue, initial_service_rate)

    discounted_return = 0.0
    current_gamma = 1.0

    rewards = np.zeros(horizon)
    throughputs = np.zeros(horizon)
    drops = np.zeros(horizon)

    total_queue_occupancy_per_user = np.zeros(N)

    for t in range(horizon):

        queues = [state[user_id][0] for user_id in range(N)]
        channels = [state[user_id][1] for user_id in range(N)]

        # Beam oracle action
        selected_user = beam_oracle_action(
            state=state,
            start_slot=start_slot,
            t=t,
            remaining_slots=horizon - t,
            lookahead_horizon=lookahead_horizon,
            beam_widths=beam_widths
        )

        # Real transmission
        service_rate = channels[selected_user]
        packets_to_send = min(queues[selected_user], service_rate)

        queues[selected_user] -= packets_to_send

        # Real random arrivals and drops
        dropped_packets = 0

        for i in range(N):
            if np.random.random() < arrival_probs[i]:
                if queues[i] < C:
                    queues[i] += 1
                else:
                    dropped_packets += 1

        # Real reward
        reward = packets_to_send - lamb * dropped_packets

        discounted_return += current_gamma * reward

        rewards[t] = reward
        throughputs[t] = packets_to_send
        drops[t] = dropped_packets

        total_queue_occupancy_per_user += np.array(queues)

        # Update channel states from trace
        next_channels = []

        for i in range(N):
            next_channel = rates[start_slot + t + 1, i]
            next_channels.append(next_channel)

        state = {
            user_id: (queues[user_id], next_channels[user_id])
            for user_id in range(N)
        }

        current_gamma *= gamma

    mean_queue_occupancy_per_user = total_queue_occupancy_per_user / horizon

    return discounted_return, rewards, throughputs, drops, mean_queue_occupancy_per_user


def run_beam_oracle(horizon, lookahead_horizon, beam_widths, num_episodes_beam):

    returns = []
    rewards_all = np.zeros((num_episodes_beam, horizon))
    throughputs_all = np.zeros((num_episodes_beam, horizon))
    drops_all = np.zeros((num_episodes_beam, horizon))
    queue_occupancies_all = np.zeros((num_episodes_beam, N))

    policy_name = f"Beam oracle L={lookahead_horizon}, B={beam_widths}"

    print(f"\n{policy_name}")
    print("-" * 30)
    print()

    for ep in range(num_episodes_beam):

        print(f"\r{policy_name}: Episode {ep+1}/{num_episodes_beam}", end="", flush=True)

        ret, rewards, throughputs, drops, queue_occ = simulate_episode_beam_oracle(
            horizon=horizon,
            lookahead_horizon=lookahead_horizon,
            beam_widths=beam_widths
        )

        returns.append(ret)
        rewards_all[ep] = rewards
        throughputs_all[ep] = throughputs
        drops_all[ep] = drops
        queue_occupancies_all[ep] = queue_occ

    print()

    mean_return = np.mean(returns)
    mean_rewards = rewards_all.mean(axis=0)
    mean_throughputs = throughputs_all.mean(axis=0)
    mean_drops = drops_all.mean(axis=0)

    mean_queue_occupancy_per_user = queue_occupancies_all.mean(axis=0)
    mean_queue_occupancy_all_users = np.mean(mean_queue_occupancy_per_user)

    print(f"{policy_name} mean discounted return:", mean_return)
    print(f"{policy_name} mean reward per round:", np.mean(mean_rewards))
    print(f"{policy_name} mean throughput per round:", np.mean(mean_throughputs))
    print(f"{policy_name} mean drops per round:", np.mean(mean_drops))
    print(f"{policy_name} mean queue occupancy all users:", mean_queue_occupancy_all_users)
    print()

    return (
        mean_return,
        mean_rewards,
        mean_throughputs,
        mean_drops,
        mean_queue_occupancy_per_user,
        mean_queue_occupancy_all_users
    )

mean_return_beam, mean_rewards_beam, mean_throughputs_beam, mean_drops_beam, mean_queue_occ_per_user_beam, mean_queue_occ_all_beam = run_beam_oracle(
    horizon=evaluation_horizon,
    lookahead_horizon=lookahead_horizon,
    beam_widths=beam_widths,
    num_episodes_beam=num_episodes_beam
)


# Plotting

## Reward

In [ ]:
plt.figure(figsize=(10, 6))

# DQN: evaluation points
plt.plot(
    evaluation_points_DQN,
    evaluation_rewards_DQN,
    color="orange",
    marker="o",
    markersize=3,
    linestyle="-",
    label="DQN"
)


# Flat line: Longest-queue baseline
plt.axhline(
    y=np.mean(mean_rewards_lq),
    color="magenta",
    linestyle="--",
    label="Longest-queue"
)

# Flat line: Max-weight baseline
plt.axhline(
    y=np.mean(mean_rewards_mw),
    color="brown",
    linestyle="--",
    label="Max-weight"
)

# Flat line: Greedy-throughput baseline
plt.axhline(
    y=np.mean(mean_rewards_gt),
    color="gray",
    linestyle="--",
    label="Greedy-throughput"
)

# Flat line: Best-nonempty-channel baseline
plt.axhline(
    y=np.mean(mean_rewards_bnec),
    color="black",
    linestyle="--",
    label="Best-nonempty-channel"
)


# Flat line: Beam-search lookahead oracle benchmark
plt.axhline(
    y=np.mean(mean_rewards_beam),
    color="blue",
    linestyle="--",
    label=f"Beam oracle L={lookahead_horizon}, B={beam_widths}"
)

plt.xlabel("Training episode")
plt.ylabel("Average evaluation reward per slot")
plt.title("Evaluation Reward During Training")
plt.grid(True)
plt.legend(fontsize=10)
plt.show()

## Throughput

In [ ]:
plt.figure(figsize=(10, 6))

# DQN: evaluation points
plt.plot(
    evaluation_points_DQN,
    evaluation_throughputs_DQN,
    color="orange",
    marker="o",
    markersize=3,
    linestyle="-",
    label="DQN"
)

# Flat line: Longest-queue baseline
plt.axhline(
    y=np.mean(mean_throughputs_lq),
    color="magenta",
    linestyle="--",
    label="Longest-queue"
)

# Flat line: Max-weight baseline
plt.axhline(
    y=np.mean(mean_throughputs_mw),
    color="brown",
    linestyle="--",
    label="Max-weight"
)

# Flat line: Greedy-throughput baseline
plt.axhline(
    y=np.mean(mean_throughputs_gt),
    color="gray",
    linestyle="--",
    label="Greedy-throughput"
)

# Flat line: Best-nonempty-channel baseline
plt.axhline(
    y=np.mean(mean_throughputs_bnec),
    color="black",
    linestyle="--",
    label="Best-nonempty-channel"
)

# Flat line: Beam-search lookahead oracle
plt.axhline(
    y=np.mean(mean_throughputs_beam),
    color="blue",
    linestyle="--",
    label=f"Beam oracle L={lookahead_horizon}, B={beam_widths}"
)

plt.xlabel("Training episode")
plt.ylabel("Average evaluation throughput per slot")
plt.title("Sample efficiency comparison")
plt.grid(True)
plt.legend(fontsize=10)
plt.show()

## Drops

In [ ]:
plt.figure(figsize=(10, 6))

# DQN: evaluation points
plt.plot(
    evaluation_points_DQN,
    evaluation_drops_DQN,
    color="orange",
    marker="o",
    markersize=3,
    linestyle="-",
    label="DQN"
)

# Flat line: Longest-queue baseline
plt.axhline(
    y=np.mean(mean_drops_lq),
    color="magenta",
    linestyle="--",
    label="Longest-queue"
)

# Flat line: Max-weight baseline
plt.axhline(
    y=np.mean(mean_drops_mw),
    color="brown",
    linestyle="--",
    label="Max-weight"
)

# Flat line: Greedy-throughput baseline
plt.axhline(
    y=np.mean(mean_drops_gt),
    color="gray",
    linestyle="--",
    label="Greedy-throughput"
)

# Flat line: Best-nonempty-channel baseline
plt.axhline(
    y=np.mean(mean_drops_bnec),
    color="black",
    linestyle="--",
    label="Best-nonempty-channel"
)

# Flat line: Beam-search lookahead oracle
plt.axhline(
    y=np.mean(mean_drops_beam),
    color="blue",
    linestyle="--",
    label=f"Beam oracle L={lookahead_horizon}, B={beam_widths}"
)

plt.xlabel("Training episode")
plt.ylabel("Average dropped packets per slot")
plt.title("Sample efficiency comparison")
plt.grid(True)
plt.legend(fontsize=10)
plt.show()

## Queue occupancies

In [ ]:
# Final record
final_queue_per_user = evaluation_queue_occupancies_per_user_DQN[-1]
final_queue_all_users = evaluation_queue_occupancies_all_users_DQN[-1]

# Labels
user_labels = [f"U{i:02d}" for i in range(N)]
labels = user_labels + ["All users"]

# Values
values = list(final_queue_per_user) + [final_queue_all_users]

# Colors: one color for users, a different one for the overall mean
colors = ["skyblue"] * N + ["orange"]

plt.figure(figsize=(10, 6))
plt.bar(labels, values, color=colors)

plt.xlabel("User")
plt.ylabel("Mean queue occupancy")
plt.title("Mean queue occupancy per user and overall")
plt.grid(True, axis="y")
plt.show()

## Reward / Throughput / Drops per round

In [ ]:
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm'

rounds = np.arange(1, evaluation_horizon + 1)

beam_label = f'Beam oracle L={lookahead_horizon}, B={beam_widths}'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Reward per round
axes[0].plot(rounds, mean_rewards_lq, label='Longest-queue')
axes[0].plot(rounds, mean_rewards_mw, label='Max-weight')
axes[0].plot(rounds, mean_rewards_gt, label='Greedy-throughput')
axes[0].plot(rounds, mean_rewards_bnec, label='Best-nonempty-channel')
axes[0].plot(rounds, evaluation_rewards_per_slot_DQN[-1], label='DQN')
axes[0].plot(rounds, mean_rewards_beam, label=beam_label)

axes[0].set_xlabel('Round')
axes[0].set_ylabel('Reward per round')
axes[0].set_title('Reward per round')
axes[0].grid(True)
axes[0].legend()

# Throughput per round
axes[1].plot(rounds, mean_throughputs_lq, label='Longest-queue')
axes[1].plot(rounds, mean_throughputs_mw, label='Max-weight')
axes[1].plot(rounds, mean_throughputs_gt, label='Greedy-throughput')
axes[1].plot(rounds, mean_throughputs_bnec, label='Best-nonempty-channel')
axes[1].plot(rounds, evaluation_throughputs_per_slot_DQN[-1], label='DQN')
axes[1].plot(rounds, mean_throughputs_beam, label=beam_label)

axes[1].set_xlabel('Round')
axes[1].set_ylabel('Throughput per round')
axes[1].set_title('Throughput per round')
axes[1].grid(True)
axes[1].legend()

# Drops per round
axes[2].plot(rounds, mean_drops_lq, label='Longest-queue')
axes[2].plot(rounds, mean_drops_mw, label='Max-weight')
axes[2].plot(rounds, mean_drops_gt, label='Greedy-throughput')
axes[2].plot(rounds, mean_drops_bnec, label='Best-nonempty-channel')
axes[2].plot(rounds, evaluation_drops_per_slot_DQN[-1], label='DQN')
axes[2].plot(rounds, mean_drops_beam, label=beam_label)

axes[2].set_xlabel('Round')
axes[2].set_ylabel('Dropped packets per round')
axes[2].set_title('Dropped packets per round')
axes[2].grid(True)
axes[2].legend()

plt.tight_layout()
plt.show()